# FerroMobile: DQN training on Kaggle with durable checkpoints

Run with **Save Version > Save & Run All (Commit)**. The run then happens on Kaggle's servers: closing the browser, losing wifi or switching off the computer does not stop it.

Every 25 episodes (and before the 12-hour session limit) the whole run folder (checkpoint, replay buffer, logs, console output) is sent as a new version of a private Kaggle dataset `fm-<run name>`. The next commit downloads the latest version and resumes exactly where the previous session stopped.

Before the first run (once):
1. Settings: Accelerator **GPU T4**, Internet **on**.
2. Add-ons > Secrets: add `KAGGLE_USERNAME` (your Kaggle user name) and `KAGGLE_KEY` (from kaggle.com > Settings > API > Create New Token, the `key` field of kaggle.json). Tick both secrets for this notebook.
3. Add Input: your DEM dataset (`ferromobile-dem`, file `eu_dem_courpiere_ambert.tif`).

To continue a run that paused at the session limit, just run **Save & Run All (Commit)** again with the same settings in cell 1.

In [ ]:
# 1. Run settings: the only cell to edit. Keep the same values to resume a run.
import os
RUN = {
    "FM_EPISODES": "1000",        # 1000 for the main runs, 300 for the reference configuration
    "FM_W_W": "5",                # 5 for the main runs, 1 for the reference configuration
    "FM_ARCH": "dueling",         # "dueling" or "vanilla"
    "FM_EXPLORATION": "softmax",  # "softmax" (Boltzmann) or "uniforme" (ablation)
    "FM_SEED": "0",               # 0, or 1 for the second Dueling seed
    "FM_FOLD": "0",               # 0 = whole line; 1 to 4 = spatial cross-validation fold (Experiment C)
    "FM_MAX_HOURS": "11",         # stop cleanly before Kaggle's 12-hour limit
    "FM_STOP_COVERED": "0",       # 1: STOP allowed only at 0 white zones
    "FM_INIT_FROM": "",           # fine-tuning: path of a dernier.pt whose weights start the run
}
BRANCHE = "claude/project-thread-snbehl"   # branch with the corrected code (use "main" once merged)
os.environ.update(RUN)
os.environ["FM_SYNC"] = "1"
print(RUN)

In [ ]:
# 2. Code, system libraries and crc-covlib (about 2 minutes)
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("ferromobile_project"):
    subprocess.run(["git", "clone", "--branch", BRANCHE, "--depth", "1",
                    "https://github.com/meyssembouzaien/ferromobile_project.git"], check=True)
os.chdir("/kaggle/working/ferromobile_project")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run("apt-get -qq update && apt-get -qq install -y libtiff-dev libgeographic-dev libgeographic19 > /dev/null",
               shell=True, check=True)
subprocess.run("pip install -q ./crc-covlib/python-wrapper && pip install -q --upgrade kaggle", shell=True, check=True)

# Check that the native library finds all its dependencies
lib = "crc-covlib/python-wrapper/crc_covlib/bin_64bit/libcrc-covlib.so"
manquant = [l for l in subprocess.run(["ldd", lib], capture_output=True, text=True).stdout.splitlines() if "not found" in l]
assert not manquant, f"Missing libraries: {manquant}"
print("crc-covlib: all dependencies found")


In [ ]:
# 3. DEM from the attached dataset
import glob, os, shutil
dem = glob.glob("/kaggle/input/**/eu_dem_courpiere_ambert.tif", recursive=True)
assert dem, "Add the DEM dataset as an input (Add Input > ferromobile-dem)"
os.makedirs("data/raw/terrain", exist_ok=True)
shutil.copy(dem[0], "data/raw/terrain/eu_dem_courpiere_ambert.tif")
print("DEM:", dem[0])

In [ ]:
# 4. Smoke tests: stop here if one fails
%cd /kaggle/working/ferromobile_project
!python placement/grid_config.py
!python placement/test_cost_model.py
!python placement/test_utility.py
!python placement/test_r_fin.py
!python -c "import crc_covlib.simulation as c; s = c.Simulation(); print('crc-covlib simulation: OK'); s.Release()"


In [ ]:
# 5. Kaggle API credentials for the durable save
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY")
    print("Kaggle secrets found: the run is saved to the private dataset after every checkpoint.")
except Exception as e:
    print("WARNING: no Kaggle secrets (", e, "). Checkpoints are only saved locally and are lost if the session stops.")

In [ ]:
# 6. Resume from a previous commit's output (Add Input > Your Work > this notebook)
# Copies the most advanced checkpoint found in /kaggle/input into checkpoints/
import glob, json, os, shutil
for etat in glob.glob("/kaggle/input/**/checkpoints/*/run_etat.json", recursive=True):
    source = os.path.dirname(etat)
    cible = os.path.join("/kaggle/working/ferromobile_project/checkpoints", os.path.basename(source))
    ep_source = json.load(open(etat))["episode"]
    ep_local = json.load(open(os.path.join(cible, "run_etat.json")))["episode"] if os.path.exists(os.path.join(cible, "run_etat.json")) else -1
    if ep_source > ep_local:
        os.makedirs(cible, exist_ok=True)
        for f in os.listdir(source):   # plain copies: files from /kaggle/input are read-only
            shutil.copyfile(os.path.join(source, f), os.path.join(cible, f))
        print(f"Checkpoint restored: {os.path.basename(source)}, episode {ep_source}")


In [ ]:
# 7. Training (resumes automatically from the latest saved version)
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
os.environ["CRC_COVLIB_PATH"] = "/kaggle/working/ferromobile_project/crc-covlib/python-wrapper"
os.environ["PYTHONUNBUFFERED"] = "1"
!cd /kaggle/working/ferromobile_project/placement && python train_dqn.py

In [ ]:
# 8. Where the logs are (also in the private dataset fm-<run name>)
import glob
for f in sorted(glob.glob("/kaggle/working/ferromobile_project/checkpoints/*/*")):
    if f.endswith((".csv", ".json", ".log")):
        print(f)